In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

# 1. Configuració de mètriques
FEATURES_BY_POS = {
    'df': ['Tkl', 'TklW', 'Int', 'Blocks', 'Clr', 'Won', 'Won%', 'Def 3rd', 'Mid 3rd', 'PrgP', 'PrgDist', 'Cmp%'],
    'mf': ['Pass', 'Cmp%', 'KP', 'xA', 'xAG', 'PPA', 'PrgP', 'PrgC', 'Mid 3rd', 'Att 3rd', 'Recov', 'Sw', 'SCA'],
    'fw': ['Gls', 'Sh', 'SoT', 'G/Sh', 'xG', 'npxG', 'Att Pen', 'Att 3rd', 'PrgR', 'SCA', 'Ast', 'CPA'],
    'gk': ['Cmp%', 'Pass', 'PrgDist', 'Touches', 'PPM']
}

# 2. Diccionari de Traducció (Ajusta els IDs segons els teus resultats)
NOMS_PERFILS = {
    'df': {0: 'Mur Defensiu', 1: 'Central Associatiu', 2: 'Lateral Ofensiu'},
    'mf': {0: 'Pivot Defensiu', 1: 'Organitzador', 2: 'Creatiu/Finalitzador'},
    'fw': {0: 'Finalitzador/Àrea', 1: 'Extrem Desequilibrant', 2: 'Fals 9/Enllaç'},
    'gk': {0: 'Porter Líbero', 1: 'Porter Sota Pals', 2: 'Porter Distribuidor'}
}

# 3. Carregar dades
df_raw = pd.read_csv('../data/processed/final_dataset_cleaned.csv')
df_raw['Pos_Main'] = df_raw['Pos'].str.split(',').str[0].str.lower()

final_dfs = []

# 4. Processament
for pos, metrics in FEATURES_BY_POS.items():
    pos_df = df_raw[df_raw['Pos_Main'] == pos].copy()
    if len(pos_df) < 15: continue

    X = pos_df[metrics].fillna(0)
    X_scaled = StandardScaler().fit_transform(X)

    # Execució GMM
    gmm = GaussianMixture(n_components=3, random_state=42, n_init=10)
    pos_df['Cluster_ID'] = gmm.fit_predict(X_scaled)
    
    # Probabilitats i Puresa
    probs = gmm.predict_proba(X_scaled)
    pos_df['Puresa_Perfil'] = probs.max(axis=1)
    pos_df['Es_Hibrid'] = np.where(pos_df['Puresa_Perfil'] < 0.6, 'Sí', 'No')

    # Mapeig de noms segons el diccionari
    pos_df['Perfil_Nom'] = pos_df['Cluster_ID'].map(NOMS_PERFILS[pos])

    final_dfs.append(pos_df)

# 5. Exportació Final
df_final = pd.concat(final_dfs, ignore_index=True)
df_final.to_csv('informe_final_scouting.csv', index=False)

print("✅ Informe generat: 'informe_final_scouting.csv'")
# Mostrar un tastet del resultat
print(df_final[['Player', 'Pos_Main', 'Perfil_Nom', 'Puresa_Perfil']].sample(10))

C:\Users\yerayhurdra\AppData\Local\Temp\ipykernel_2452\2522230801.py:23: DtypeWarning: Columns (112) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../data/processed/final_dataset_cleaned.csv')


✅ Informe generat: 'informe_final_scouting.csv'
                      Player Pos_Main             Perfil_Nom  Puresa_Perfil
13532        riccardo sottil       fw          Fals 9/Enllaç       0.998493
9941          yangel herrera       mf   Creatiu/Finalitzador       1.000000
584        lukas klostermann       df           Mur Defensiu       1.000000
13466         andrea belotti       fw  Extrem Desequilibrant       1.000000
9318         piotr zieliński       mf         Pivot Defensiu       1.000000
1661      geoffrey kondogbia       df     Central Associatiu       1.000000
8639   youssouf ndayishimiye       mf   Creatiu/Finalitzador       0.999991
15364           rui patrício       gk       Porter Sota Pals       0.999992
12724        mostafa mohamed       fw  Extrem Desequilibrant       0.687665
10108          ismaïl bouneb       mf           Organitzador       1.000000
